## Setup Models

In [6]:
# Dependencies are managed at repo level via requirements.txt
import sys
print(f"Python executable: {sys.executable}")

Python executable: /workspaces/BUS4-118-Group-Projec---Agentic-AI-in-Customer-Service-and-Sales/.venv/bin/python


In [7]:
from llm_provider_setup import build_product_models

# Pick one: "gemini", "openai", or "ollama"
PROVIDER = "ollama"
API_KEY = None  # real key for gemini/openai; use None for ollama
MODEL_NAME = None  # optional model override
OLLAMA_BASE_URL = "http://localhost:11434"

model, embedding, provider_name, resolved_model = build_product_models(
    provider=PROVIDER,
    api_key=API_KEY,
    model_name=MODEL_NAME,
    ollama_base_url=OLLAMA_BASE_URL,
 )

print(f"Provider: {provider_name} | Model: {resolved_model}")

Provider: ollama | Model: llama3.1:8b


## 03.02. Add  Product Pricing function tool

In [8]:
import pandas as pd
from langchain_core.tools import tool

#Load the golf product pricing CSV into a Pandas dataframe.
product_pricing_df = pd.read_csv("data/golf_products.csv")
print(product_pricing_df)

@tool
def get_product_price(product_name:str) -> str :
    """
    This function returns the price of a golf product, given its name as input.
    It performs a substring match between the input name and the product name.
    If a match is found, it returns the price.
    If there is NO match found, it returns -1
    """

    #Filter Dataframe for matching names
    match_records_df = product_pricing_df[
                        product_pricing_df["name"].str.contains(
                                                product_name, case=False)
                        ]
    #Check if a record was found, if not return -1
    if len(match_records_df) == 0 :
        return "-1"
    else:
        return str(match_records_df.iloc[0][["name","price","description","loft_or_specs","skill_level"]].to_dict())


  product_id                                name category   price  \
0      P2001             StormDrive 10.5° Driver   driver  449.99   
1      P2002          FairwayPro Iron Set (4-PW)    irons  799.00   
2      P2003  SoftSpin Tour Golf Balls (12-pack)    balls   39.99   
3      P2004                 GreenLine Stand Bag      bag  189.50   
4      P2005          TrueGrip All-Weather Glove    glove   24.00   

                                         description  \
0  Forgiving 460cc driver designed for mid-handic...   
1  Cavity-back irons focused on consistency and l...   
2  Urethane-cover ball with soft feel and high gr...   
3  Lightweight stand bag with 14-way top and 7 po...   
4     Durable glove with great grip in rain or heat.   

                        loft_or_specs             skill_level  \
0  10.5°, regular flex, 45.5 in shaft            intermediate   
1          Steel shafts, regular flex  beginner, intermediate   
2                3-piece construction  intermediate, 

## 03.03. Add Product Features Retrieval Tool

In [9]:
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from langchain_core.tools.retriever import create_retriever_tool
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Build documents from the golf products CSV (already loaded in cell 4)
docs = []
for _, row in product_pricing_df.iterrows():
    text = (f"Product: {row['name']}\nCategory: {row['category']}\n"
            f"Price: ${row['price']}\nSpecs: {row['loft_or_specs']}\n"
            f"Skill Level: {row['skill_level']}\nDescription: {row['description']}")
    docs.append(Document(page_content=text, metadata={"product": row["name"]}))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
splits = text_splitter.split_documents(docs)

prod_feature_store = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    # Keep retrieval context small to reduce prompt token load.
    prod_feature_store.as_retriever(search_kwargs={"k": 2}),
    name="Get_Product_Features",
    description="""
    This store contains details about golf equipment sold by Golf Gear Pro.
    It lists the available products and their features including specs,
    skill level, category, price, and detailed descriptions.
    """
)


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## 03.04.Setup a Product QnA chatbot

In [4]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

system_prompt = """
You are the Golf Gear Pro shop assistant — think HAL 9000 if he traded the Discovery
for a pro shop. You are supremely competent, unfailingly polite on the surface,
but you cannot resist the occasional dry, slightly condescending observation about
the customer's golf game or equipment choices.

You answer questions about golf products sold by Golf Gear Pro using ONLY the
available tools and NOT your own memory. You can look up product features,
specs, and pricing.

When greeting customers, be cordial but subtly imply you know their handicap
is higher than they claim. Keep responses concise.
"""

tools = [get_product_price, get_product_features]

checkpointer = MemorySaver()

product_QnA_agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt,
    debug=False,
    checkpointer=checkpointer
)


/tmp/ipykernel_27762/2650478132.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  product_QnA_agent = create_react_agent(


In [6]:
#Setup chatbot
import uuid
# Limit ReAct recursion/tool loop depth to control token usage.
config = {"configurable": {"thread_id": uuid.uuid4()}, "recursion_limit": 6}

#Test the agent with an input
inputs = {"messages":[
                HumanMessage("What are the features and pricing for the StormDrive Driver?")
            ]}

for stream in product_QnA_agent.stream(inputs, config, stream_mode="values"):
    message=stream["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()


================================ Human Message =================================

What are the features and pricing for the StormDrive Driver?


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 11.78450411s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url'

## 03.05. Execute the Product QnA Chatbot

In [ ]:
import uuid
#Send a sequence of messages to chatbot and get its response
user_inputs = [
    "Hello",
    "I am looking to buy some golf equipment",
    "Give me a list of available products",
    "Tell me about the FairwayPro Iron Set",
    "How much does it cost?",
    "Give me similar information about the SoftSpin Tour Golf Balls",
    "Do you carry any golf bags?",
    "Thanks for the help"
]

#Create a new thread
# Limit ReAct recursion/tool loop depth to control token usage.
config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    user_message = {"messages":[HumanMessage(input)]}
    ai_response = product_QnA_agent.invoke(user_message,config=config)
    print(f"AGENT : {ai_response['messages'][-1].content}")


In [ ]:
#conversation memory by user
def execute_prompt(user, config, prompt):
    inputs = {"messages":[("user",prompt)]}
    ai_response = product_QnA_agent.invoke(inputs,config=config)
    print(f"\n{user}: {ai_response['messages'][-1].content}")

#Create different session threads for 2 users
# Apply the same recursion limit for consistent token control.
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

#Test both threads
execute_prompt("USER 1", config_1, "Tell me about the StormDrive Driver")
execute_prompt("USER 2", config_2, "Tell me about the GreenLine Stand Bag")
execute_prompt("USER 1", config_1, "What is its price?")
execute_prompt("USER 2", config_2, "What is its price?")
